# SHIELD — RF with30 financial-statement candidates versus20 historical inputs

**Question:** does the broader pool of30 dataset features potentially calculable from ordinary SME annual-account amounts preserve more predictive information than our seven-input trial, compared with the matched20-input baseline?

One self-contained notebook; two models per run. **Stratified80:20,10-fold ordinary training CV,seed42,threshold0.5.** Start with SMOTE0.25, change it to1.0 for the second run. Both settings save separately.

**No imputation, new IG selection, hyperparameter search or threshold tuning.** All30 candidates are used in the30-input arm. The20-input arm uses the historical predefined list. No prototype/model deployment changes.

## What 'financial-statement candidates' means
All30 names are actual columns in the local dataset. We identified their underlying accounting amounts in reporting structures/examples. This does **not** mean every SME must print each ratio, every company has all required breakdowns, or all exact donor formulas have been verified. The target population for the availability research is incorporated SMEs supplying complete annual accounts and relevant notes.

The calculations in the evidence manifest are conceptual, not implemented here. This notebook selects existing dataset columns. It does not accept raw financial statements, reconstruct raw ratios, invent unavailable company amounts, or establish the original donor's raw-to-model transformation. That transformation, exact definitions and real SME availability remain deployment gates.

Some columns may overlap (e.g. current-liability ratios); do not assume30 independent signals. Potential duplicates are deliberately retained for this initial fixed-pool experiment and labelled in the manifest.


## 1. Install dependencies
If Colab imported a different modelling version before installation, restart the runtime after installing and continue from settings.

In [ ]:
%pip install -q scikit-learn==1.9.0 imbalanced-learn==0.14.2 pandas numpy matplotlib openpyxl


## 2. Settings
Change only SMOTE_RATIO to1.0 for the second experiment, then rerun from this cell onward. Use original unresampled data. Update Drive paths if needed.

In [ ]:
from pathlib import Path
SMOTE_RATIO=0.25           # Second run: change to 1.0
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE=Path('/content/drive/MyDrive/SHIELD_Project')
except ImportError:
    BASE=Path.cwd()
    if BASE.name=='notebooks': BASE=BASE.parent
RAW_PATH=BASE/'data'/'bankruptcy_italian_companies_2023.xlsx'
OUTPUT_BASE=BASE/'reports'/'rf_financial_statement30_vs_twenty'
print('Data:',RAW_PATH)
print('SMOTE ratio:',SMOTE_RATIO)


## 3. Fixed candidates and identical model settings
RF:200trees,max_depth10,min_samples_split2,min_samples_leaf1,max_features sqrt,no class weights. SMOTE:k_neighbors5,random_state42. No new scaling, matching the controlled historical RF+SMOTE setup. With a different number of input columns, sqrt naturally changes the absolute split-feature count.

SMOTE is fitted **only on each training fold**; validation/test rows retain their original class distribution. The test set has been inspected in earlier experiments, so comparisons are exploratory. The accounting shortlist was not selected by optimising this test set, but it intersects the historical IG selection and inherits its limitations.

The baseline is refitted in the same environment; use its fresh results for comparison. Same RF parameters are retained for both ratios rather than reproducing differently tuned historical winners.


In [ ]:
import hashlib, json, importlib.metadata
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline

BASELINE_FEATURES=['ROA(C) before interest and depreciation before interest', 'ROA(A) before interest and % after tax', 'ROA(B) before interest and depreciation after tax', 'Continuous interest rate (after tax)', 'Net Value Per Share (B)', 'Net Value Per Share (A)', 'Persistent EPS in the Last Four Seasons', 'Per Share Net profit before tax', 'Interest Expense Ratio', 'Debt ratio %', 'Net worth/Assets', 'Borrowing dependency', 'Net profit before tax/Paid-in capital', 'Retained Earnings to Total Assets', 'Net Income to Total Assets', "Net Income to Stockholder's Equity", 'Liability to Equity', 'Degree of Financial Leverage (DFL)', 'Interest Coverage Ratio (Interest expense to EBIT)', 'Equity to Liability']
STATEMENT_FEATURES=['Operating Gross Margin', 'Operating Profit Rate', 'Pre-tax net Interest Rate', 'After-tax net Interest Rate', 'Current Ratio', 'Debt ratio %', 'Net worth/Assets', 'Working Capital to Total Assets', 'Current Assets/Total Assets', 'Cash/Total Assets', 'Cash/Current Liability', 'Current Liability to Assets', 'Inventory/Working Capital', 'Inventory/Current Liability', 'Current Liabilities/Liability', 'Working Capital/Equity', 'Current Liabilities/Equity', 'Long-term Liability to Current Assets', 'Retained Earnings to Total Assets', 'Current Liability to Liability', 'Current Liability to Equity', 'Equity to Long-term Liability', 'CFO to Assets', 'Current Liability to Current Assets', 'Liability-Assets Flag', 'Net Income to Total Assets', 'Gross Profit to Sales', "Net Income to Stockholder's Equity", 'Liability to Equity', 'Equity to Liability']
FEATURES=['ROA(C) before interest and depreciation before interest', 'ROA(A) before interest and % after tax', 'ROA(B) before interest and depreciation after tax', 'Continuous interest rate (after tax)', 'Net Value Per Share (B)', 'Net Value Per Share (A)', 'Persistent EPS in the Last Four Seasons', 'Per Share Net profit before tax', 'Interest Expense Ratio', 'Debt ratio %', 'Net worth/Assets', 'Borrowing dependency', 'Net profit before tax/Paid-in capital', 'Retained Earnings to Total Assets', 'Net Income to Total Assets', "Net Income to Stockholder's Equity", 'Liability to Equity', 'Degree of Financial Leverage (DFL)', 'Interest Coverage Ratio (Interest expense to EBIT)', 'Equity to Liability', 'Operating Gross Margin', 'Operating Profit Rate', 'Pre-tax net Interest Rate', 'After-tax net Interest Rate', 'Current Ratio', 'Working Capital to Total Assets', 'Current Assets/Total Assets', 'Cash/Total Assets', 'Cash/Current Liability', 'Current Liability to Assets', 'Inventory/Working Capital', 'Inventory/Current Liability', 'Current Liabilities/Liability', 'Working Capital/Equity', 'Current Liabilities/Equity', 'Long-term Liability to Current Assets', 'Current Liability to Liability', 'Current Liability to Equity', 'Equity to Long-term Liability', 'CFO to Assets', 'Current Liability to Current Assets', 'Liability-Assets Flag', 'Gross Profit to Sales']  # Union for input validation only; each model uses its own list.
STATEMENT_MANIFEST=[{'number': 4, 'feature': 'Operating Gross Margin', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Revenue; cost of sales or gross profit', 'conceptual_calculation': 'Gross profit / revenue', 'time_requirement': 'One annual period', 'caution': 'Confirm gross-profit scope.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 6, 'feature': 'Operating Profit Rate', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Operating profit; revenue', 'conceptual_calculation': 'Operating profit / revenue', 'time_requirement': 'One annual period', 'caution': 'Use operating profit, not profit after tax; consistent expense classification.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 7, 'feature': 'Pre-tax net Interest Rate', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Profit before tax; revenue', 'conceptual_calculation': 'Profit before tax / revenue', 'time_requirement': 'One annual period', 'caution': 'Label appears to mean pretax margin, not bank interest rate.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 8, 'feature': 'After-tax net Interest Rate', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Profit after tax; revenue', 'conceptual_calculation': 'Profit after tax / revenue', 'time_requirement': 'One annual period', 'caution': 'Label appears to mean after-tax margin, not bank interest rate.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 33, 'feature': 'Current Ratio', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Current assets; current liabilities', 'conceptual_calculation': 'Current assets / current liabilities', 'time_requirement': 'Same reporting date', 'caution': 'Requires current/non-current presentation or equivalent breakdown.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 37, 'feature': 'Debt ratio %', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': True, 'required_company_information': 'Total liabilities; total assets', 'conceptual_calculation': 'Total liabilities / total assets', 'time_requirement': 'Same reporting date', 'caution': 'Confirm percent/fraction convention; not interest-bearing debt only.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 38, 'feature': 'Net worth/Assets', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': True, 'required_company_information': 'Total equity; total assets', 'conceptual_calculation': 'Equity / total assets', 'time_requirement': 'Same reporting date', 'caution': 'Confirm total equity versus parent-only equity.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 54, 'feature': 'Working Capital to Total Assets', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Current assets; current liabilities; total assets', 'conceptual_calculation': '(Current assets - current liabilities) / assets', 'time_requirement': 'Same reporting date', 'caution': 'Working capital can be negative; do not clamp.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 56, 'feature': 'Current Assets/Total Assets', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Current assets; total assets', 'conceptual_calculation': 'Current assets / assets', 'time_requirement': 'Same reporting date', 'caution': 'Consistent entity scope.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 57, 'feature': 'Cash/Total Assets', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Eligible cash/cash equivalents; total assets', 'conceptual_calculation': 'Cash / assets', 'time_requirement': 'Same reporting date', 'caution': 'Confirm whether equivalents/restricted deposits belong in numerator.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 59, 'feature': 'Cash/Current Liability', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Eligible cash/cash equivalents; current liabilities', 'conceptual_calculation': 'Cash / current liabilities', 'time_requirement': 'Same reporting date', 'caution': 'Confirm cash definition and nonzero denominator.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 60, 'feature': 'Current Liability to Assets', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Current liabilities; total assets', 'conceptual_calculation': 'Current liabilities / assets', 'time_requirement': 'Same reporting date', 'caution': 'Current/non-current classifications needed.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 62, 'feature': 'Inventory/Working Capital', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Inventory; current assets; current liabilities', 'conceptual_calculation': 'Inventory / (current assets - current liabilities)', 'time_requirement': 'Same reporting date', 'caution': 'Undefined at zero working capital; negative values may be meaningful.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 63, 'feature': 'Inventory/Current Liability', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Inventory; current liabilities', 'conceptual_calculation': 'Inventory / current liabilities', 'time_requirement': 'Same reporting date', 'caution': 'Do not assume zero inventory for service firms without checking.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 64, 'feature': 'Current Liabilities/Liability', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Current liabilities; total liabilities', 'conceptual_calculation': 'Current liabilities / total liabilities', 'time_requirement': 'Same reporting date', 'caution': 'Potential duplicate of Current Liability to Liability; definitions must match.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 65, 'feature': 'Working Capital/Equity', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Current assets; current liabilities; equity', 'conceptual_calculation': '(Current assets - current liabilities) / equity', 'time_requirement': 'Same reporting date', 'caution': 'Zero or negative equity needs explicit treatment.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 66, 'feature': 'Current Liabilities/Equity', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Current liabilities; equity', 'conceptual_calculation': 'Current liabilities / equity', 'time_requirement': 'Same reporting date', 'caution': 'Potential duplicate of Current Liability to Equity.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 67, 'feature': 'Long-term Liability to Current Assets', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Non-current liabilities; current assets', 'conceptual_calculation': 'Long-term liabilities / current assets', 'time_requirement': 'Same reporting date', 'caution': 'Verify long-term liability scope.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 68, 'feature': 'Retained Earnings to Total Assets', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': True, 'required_company_information': 'Retained earnings/accumulated losses; assets', 'conceptual_calculation': 'Retained earnings / assets', 'time_requirement': 'Same reporting date', 'caution': 'Retained earnings is not current-year profit or total reserves.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 77, 'feature': 'Current Liability to Liability', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Current liabilities; total liabilities', 'conceptual_calculation': 'Current liabilities / total liabilities', 'time_requirement': 'Same reporting date', 'caution': 'Potential duplicate of Current Liabilities/Liability.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 78, 'feature': 'Current Liability to Equity', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Current liabilities; equity', 'conceptual_calculation': 'Current liabilities / equity', 'time_requirement': 'Same reporting date', 'caution': 'Potential duplicate of Current Liabilities/Equity.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 79, 'feature': 'Equity to Long-term Liability', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Equity; non-current liabilities', 'conceptual_calculation': 'Equity / long-term liabilities', 'time_requirement': 'Same reporting date', 'caution': 'Undefined if eligible long-term liability is zero.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 82, 'feature': 'CFO to Assets', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Net cash from operating activities; assets', 'conceptual_calculation': 'CFO / assets', 'time_requirement': 'Annual cash flow and specified asset balance', 'caution': 'Confirm average/end assets; cash flows are shown in MPERS examples.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 84, 'feature': 'Current Liability to Current Assets', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Current liabilities; current assets', 'conceptual_calculation': 'Current liabilities / current assets', 'time_requirement': 'Same reporting date', 'caution': 'Not a growth or cash-flow measure.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 85, 'feature': 'Liability-Assets Flag', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Total liabilities; total assets', 'conceptual_calculation': 'Indicator based on liabilities exceeding assets', 'time_requirement': 'Same reporting date', 'caution': 'Confirm encoded direction (0/1); derived flag, not a printed statement field.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 86, 'feature': 'Net Income to Total Assets', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': True, 'required_company_information': 'Net income; assets', 'conceptual_calculation': 'Net income / assets', 'time_requirement': 'Annual flow and specified asset balance', 'caution': 'Confirm average/end assets and continuing/total profit scope.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 89, 'feature': 'Gross Profit to Sales', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': False, 'required_company_information': 'Gross profit; sales', 'conceptual_calculation': 'Gross profit / sales', 'time_requirement': 'Matching annual period', 'caution': 'Potential overlap with Operating Gross Margin; definitions not assumed identical.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 90, 'feature': "Net Income to Stockholder's Equity", 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': True, 'required_company_information': 'Net income; shareholders equity', 'conceptual_calculation': 'Net income / eligible equity', 'time_requirement': 'Annual flow and specified equity balance', 'caution': 'Confirm ending/average and parent/total equity.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 91, 'feature': 'Liability to Equity', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': True, 'required_company_information': 'Total liabilities; eligible equity', 'conceptual_calculation': 'Liabilities / equity', 'time_requirement': 'Same reporting date', 'caution': 'Check absolute-equity/sign conventions and scope.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}, {'number': 95, 'feature': 'Equity to Liability', 'category': 'A', 'category_label': 'Annual-account calculator candidate', 'top20': True, 'required_company_information': 'Equity; total liabilities', 'conceptual_calculation': 'Equity / liabilities', 'time_requirement': 'Same reporting date', 'caution': 'Undefined at zero liabilities; reciprocal only under matching raw conventions.', 'directly_printed_ratio_guaranteed': False, 'original_model_scale_verified': False, 'constant_in_local_data': False, 'source_refs': ['SSM', 'NK', 'KPMG', 'IFRS', 'UCI'], 'evidence_level': 'Analyst feasibility assessment; not a confirmed exact donor-to-MPERS match'}]

def scenarios():
    return {'baseline_20':BASELINE_FEATURES.copy(),'financial_statement_30':STATEMENT_FEATURES.copy()}

def make_model(ratio,small=False):
    return Pipeline([('smote',SMOTE(sampling_strategy=ratio,k_neighbors=5,random_state=42)),
        ('model',RandomForestClassifier(n_estimators=5 if small else 200,max_depth=10,
        min_samples_split=2,min_samples_leaf=1,max_features='sqrt',class_weight=None,
        random_state=42,n_jobs=2))])

def score_metrics(y,score):
    pred=(np.asarray(score)>=.5).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {'Accuracy (%)':100*accuracy_score(y,pred),
        'Precision (%)':100*precision_score(y,pred,zero_division=0),
        'Recall (%)':100*recall_score(y,pred,zero_division=0),
        'F1-Score (%)':100*f1_score(y,pred,zero_division=0),
        'ROC-AUC (%)':100*roc_auc_score(y,score),
        'F2 (%)':100*fbeta_score(y,pred,beta=2,zero_division=0),
        'PR-AUC (AP) (%)':100*average_precision_score(y,score),
        'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp)}

def checked_predictions(path,rows,y):
    with np.load(path,allow_pickle=False) as saved:
        np.testing.assert_array_equal(saved['row_index'],rows)
        np.testing.assert_array_equal(saved['y_true'],np.asarray(y))
        score=saved['y_score'].copy()
    if len(score)!=len(y) or not np.isfinite(score).all():
        raise ValueError('Invalid prediction checkpoint: '+str(path))
    return score


## 4. Validate source and create one matched partition
Validate the43-column union of both feature sets:20+30-7shared. The models use20 and30 respectively. Company and Bankrupt are excluded. No silent row dropping, missing-value filling or label changes. Export the accounting-candidate evidence manifest, exact parameters, data fingerprint, package versions and fold membership.

In [ ]:
raw=pd.read_excel(RAW_PATH) if RAW_PATH.suffix.lower() in ['.xlsx','.xls'] else pd.read_csv(RAW_PATH,float_precision='round_trip')
raw.columns=raw.columns.str.strip()
if raw.columns.duplicated().any(): raise ValueError('Duplicate column names')
if 'Bankrupt' not in raw: raise ValueError('Bankrupt target required')
if 'Company' in raw and raw.Company.duplicated().any(): raise ValueError('Duplicate company IDs: investigate/group before splitting')
if any(f not in raw for f in FEATURES): raise ValueError('Input does not contain all43 comparison features')
X=raw[FEATURES].apply(pd.to_numeric,errors='raise')
y=pd.to_numeric(raw.Bankrupt,errors='raise')
if X.isna().any().any() or y.isna().any() or not np.isfinite(X.to_numpy()).all():
    raise ValueError('Supply complete original data. Scenarios remove columns; no imputation is performed.')
if set(y.unique())!={0,1}: raise ValueError('Target must contain 0=Not Bankrupt and 1=Bankrupt')
y=y.astype(int)
if X.duplicated().any(): raise ValueError('Duplicate full predictor rows: investigate before splitting')
train_idx,test_idx=train_test_split(np.arange(len(y)),test_size=.20,stratify=y,random_state=42)
X_train,y_train=X.iloc[train_idx],y.iloc[train_idx]
X_test,y_test=X.iloc[test_idx],y.iloc[test_idx]
if SMOTE_RATIO not in [.25,1.0]: raise ValueError('Choose SMOTE_RATIO=0.25 or 1.0')
SCENARIOS=scenarios()
versions={p:importlib.metadata.version(p) for p in ['numpy','pandas','scikit-learn','imbalanced-learn']}
config={'version':'shield-rf-financial-statement30-v1','raw_sha256':hashlib.sha256(RAW_PATH.read_bytes()).hexdigest(),
    'feature_names':FEATURES,'scenarios':SCENARIOS,'smote_ratio':SMOTE_RATIO,
    'seed':42,'test_size':.20,'folds':10,'threshold':.5,'versions':versions,
    'rf':make_model(SMOTE_RATIO).named_steps['model'].get_params(),
    'smote_k_neighbors':5,'scaling':'none','feature_selection':'fixed historical20 versus accounting-candidate30; no IG reselection'}
serialized=json.dumps(config,sort_keys=True,default=str)
RUN_ID=hashlib.sha256(serialized.encode()).hexdigest()[:16]
OUT=OUTPUT_BASE/RUN_ID
OUT.mkdir(parents=True,exist_ok=True)
if (OUT/'run_config.json').exists() and (OUT/'run_config.json').read_text(encoding='utf-8')!=serialized:
    raise ValueError('Checkpoint settings mismatch')
(OUT/'run_config.json').write_text(serialized,encoding='utf-8')
folds=list(StratifiedKFold(n_splits=10,shuffle=True,random_state=42).split(X_train,y_train))
membership=pd.DataFrame({'row_index':np.arange(len(y)),'Bankrupt':y,
    'partition':np.where(np.isin(np.arange(len(y)),train_idx),'training','test'),'validation_fold':-1})
for fold,(_,va) in enumerate(folds,1): membership.loc[train_idx[va],'validation_fold']=fold
membership.to_csv(OUT/'partition_membership.csv',index=False)
manifest=[{'Scenario':name,'Features':len(cols),'Removed':'; '.join(c for c in BASELINE_FEATURES if c not in cols),'Added':'; '.join(c for c in cols if c not in BASELINE_FEATURES)} for name,cols in SCENARIOS.items()]
pd.DataFrame(manifest).to_csv(OUT/'scenario_manifest.csv',index=False)
print('Original training counts:',y_train.value_counts().sort_index().to_dict())
print('Original test counts:',y_test.value_counts().sort_index().to_dict())
print('Scenarios:',len(SCENARIOS),'| SMOTE ratio:',SMOTE_RATIO,'| Output:',OUT)
display(pd.DataFrame(manifest))

feature_evidence=pd.DataFrame([{'Feature':r['feature'],
    'In historical20':r['top20'],'Required accounting amounts':r['required_company_information'],
    'Candidate calculation':r['conceptual_calculation'],'Remaining definition checks':r['caution'],
    'Exact original-model scale verified':r['original_model_scale_verified']} for r in STATEMENT_MANIFEST])
feature_evidence.to_csv(OUT/'financial_statement_feature_manifest.csv',index=False)
display(feature_evidence)
print('Shared financial features:',len(set(BASELINE_FEATURES)&set(STATEMENT_FEATURES)))
print('No new IG selection. Thirty means30 dataset columns, not30 guaranteed reported ratios.')


## 5. Ordinary10-fold training CV
Two feature sets ×10 folds; each company has one out-of-fold prediction. These are fixed-parameter evaluations. Mean fold F1 and pooled OOF F1 differ. Completed fold predictions are checked/reused for identical settings; changing ratio produces a separate run folder.

In [ ]:
rows=[]
for name,columns in SCENARIOS.items():
    directory=OUT/name
    directory.mkdir(exist_ok=True)
    oof=np.full(len(y_train),np.nan)
    fold_metrics=[]
    for fold,(tr,va) in enumerate(folds,1):
        checkpoint=directory/f'fold_{fold:02d}.npz'
        if checkpoint.exists():
            scores=checked_predictions(checkpoint,train_idx[va],y_train.iloc[va])
        else:
            pipe=make_model(SMOTE_RATIO)
            pipe.fit(X_train.iloc[tr][columns],y_train.iloc[tr])
            scores=pipe.predict_proba(X_train.iloc[va][columns])[:,1]
            np.savez_compressed(checkpoint,row_index=train_idx[va],y_true=y_train.iloc[va],y_score=scores)
        oof[va]=scores
        fold_metrics.append({'Fold':fold,**score_metrics(y_train.iloc[va],scores)})
        print(name,'fold',fold,'F1:',round(fold_metrics[-1]['F1-Score (%)'],2))
    assert np.isfinite(oof).all()
    pd.DataFrame(fold_metrics).to_csv(directory/'fold_metrics.csv',index=False)
    np.savez_compressed(directory/'training_oof.npz',row_index=train_idx,y_true=y_train,y_score=oof)
    details=next(m for m in manifest if m['Scenario']==name)
    row={**details,'Model':'RandomForest','Experiment':'80-20 + CV=10',
        'Stage':'Training OOF','Split':.20,'CV':10,'SMOTE':SMOTE_RATIO,
        **score_metrics(y_train,oof),
        'CV_F1_Mean (%)':float(np.mean([m['F1-Score (%)'] for m in fold_metrics])),
        'CV_F1_SD (%)':float(np.std([m['F1-Score (%)'] for m in fold_metrics],ddof=1))}
    rows.append(row)
    pd.DataFrame(rows).to_csv(OUT/'results.csv',index=False)
    report=classification_report(y_train,(oof>=.5).astype(int),labels=[0,1],target_names=['Not Bankrupt','Bankrupt'],digits=4,zero_division=0)
    (directory/'training_oof_report.txt').write_text(report,encoding='utf-8')
print('Training CV complete. These scores become selection scores if used to choose a scenario.')
display(pd.DataFrame(rows).round(2))


## 6. Final training refits and test reports
Two more fits: one per feature set on the complete80% training portion. Total22fits per SMOTE ratio. No test resampling. Existing20/7production pipelines are not changed, and no replacement production model is exported.

In [ ]:
# Read validated OOF summary so this cell can be rerun without duplicating rows.
rows=pd.read_csv(OUT/'results.csv',float_precision='round_trip').query("Stage == 'Training OOF'").to_dict('records')
for name,columns in SCENARIOS.items():
    directory=OUT/name
    checkpoint=directory/'test_predictions.npz'
    if checkpoint.exists():
        scores=checked_predictions(checkpoint,test_idx,y_test)
    else:
        pipe=make_model(SMOTE_RATIO)
        pipe.fit(X_train[columns],y_train)
        scores=pipe.predict_proba(X_test[columns])[:,1]
        np.savez_compressed(checkpoint,row_index=test_idx,y_true=y_test,y_score=scores)
    reference=next(r for r in rows if r['Scenario']==name and r['Stage']=='Training OOF')
    row={**reference,'Stage':'Test',**score_metrics(y_test,scores)}
    rows.append(row)
    report=f'=== Classification Report: RandomForest | 80-20 + CV=10 | SMOTE={SMOTE_RATIO} | {name} ===\n'
    report+=classification_report(y_test,(scores>=.5).astype(int),labels=[0,1],target_names=['Not Bankrupt','Bankrupt'],digits=4,zero_division=0)
    report+='\n'+' | '.join(f'{k}: {row[k]:.2f}' for k in ['Accuracy (%)','Precision (%)','Recall (%)','F1-Score (%)','ROC-AUC (%)'])
    print(report)
    (directory/'test_report.txt').write_text(report,encoding='utf-8')
    fig,ax=plt.subplots(figsize=(5.4,4.5))
    ConfusionMatrixDisplay.from_predictions(y_test,(scores>=.5).astype(int),labels=[0,1],display_labels=['Not Bankrupt','Bankrupt'],ax=ax,cmap='Blues',colorbar=False)
    ax.set_title(f'{name} | SMOTE {SMOTE_RATIO}')
    fig.tight_layout()
    fig.savefig(directory/'test_confusion_matrix.svg',bbox_inches='tight')
    fig.savefig(directory/'test_confusion_matrix.png',dpi=180,bbox_inches='tight')
    plt.show()
    fig,axes=plt.subplots(1,2,figsize=(11,4.5))
    RocCurveDisplay.from_predictions(y_test,scores,ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_test,scores,ax=axes[1])
    fig.suptitle(f'{name} | test | SMOTE {SMOTE_RATIO}')
    fig.tight_layout()
    fig.savefig(directory/'test_roc_pr.png',dpi=180,bbox_inches='tight')
    fig.savefig(directory/'test_roc_pr.svg',bbox_inches='tight')
    plt.show()
    pd.DataFrame(rows).to_csv(OUT/'results.csv',index=False)
results=pd.DataFrame(rows)
(OUT/'RESULTS.txt').write_text('SHIELD: fixed20 versus financial-statement-candidate30 Random Forests; no imputation.\n'+
    '80:20 split, training CV=10, fixed parameters, seed42, threshold0.5.\n'+
    'Previously inspected holdout; exploratory results.\n\n'+results.to_string(index=False),encoding='utf-8')
display(results.round(2))


## 7. Baseline comparison
Removed/Added are relative to the historical20 inputs. Changes in percentage metrics are percentage points. Compare precision,recall,F1,missed bankruptcies and false alarms, not accuracy alone. No automatic winner is chosen from test results.

In [ ]:
comparison=results[results.Stage=='Test'].copy()
baseline=comparison[comparison.Scenario=='baseline_20'].iloc[0]
for metric in ['Precision (%)','Recall (%)','F1-Score (%)','F2 (%)','FP','FN']:
    comparison['Change '+metric]=comparison[metric]-baseline[metric]
comparison.to_csv(OUT/'test_changes_vs_baseline.csv',index=False)
# Keep original scenario order. Do not automatically pick a winner from the test set.
cols=['Scenario','Features','Removed','Added','Precision (%)','Recall (%)','F1-Score (%)','FP','FN',
    'Change Precision (%)','Change Recall (%)','Change F1-Score (%)']
display(comparison[cols].round(2))
fig,ax=plt.subplots(figsize=(12,8))
comparison.set_index('Scenario')[['Precision (%)','Recall (%)','F1-Score (%)']].plot.barh(ax=ax)
ax.set_xlabel('Score (%)'); ax.set_xlim(0,100)
ax.set_title(f'Thirty versus twenty inputs: RF | test | SMOTE {SMOTE_RATIO}')
fig.tight_layout()
fig.savefig(OUT/'test_comparison.svg',bbox_inches='tight')
fig.savefig(OUT/'test_comparison.png',dpi=180,bbox_inches='tight')
plt.show()
print('All outputs:',OUT)


## Reading results
- First compare30 with20 at the same SMOTE ratio. Then repeat with1.0. Do not assume more inputs necessarily improve precision or recall.
- Use prior seven-input results as context; this notebook trains the30 and20 arms only. Compare prior runs cautiously if package versions, data precision or settings differ.
- If the30-input model is promising, the next optional comparison is training-fold IG within the eligible pool. IG is not futile; it is simply omitted from this initial all30 comparison.
- The hypothetical30 ratios may be derived from fewer underlying financial amounts. A practical form would collect those amounts after definition/scale verification rather than assume users know every ratio.
- One of44 bankrupt test companies corresponds to2.27percentage points of recall. Independent validation is needed after selecting the final model and input policy.

## Evidence sources
[SSM MBRS FAQ v2.8,PDFp12](https://www.ssm.com.my/bm/Pages/Register_Business_Company_LLP/Company/document/FAQ-MBRS_ISSB.pdf#page=12): reporting components, not a guarantee of each ratio.
[Malaysian MPERS illustrative accounts,PDFpp21-25](https://nk.com.my/cdn/pdf/MPERS_Illustrative_Financial_Statements.pdf#page=21): example amounts and notes, not proof every business supplies them.
[IFRS illustrative examples](https://www.ifrs.org/issued-standards/ifrs-taxonomy/ifrs-taxonomy-illustrative-examples/): supplementary accounting-format evidence; user's supplied iXBRL example was inspected in research.
[UCI dataset](https://archive.ics.uci.edu/dataset/572/taiwanese+bankruptcy+prediction): historical financial-ratio benchmark; local filename alone does not establish country provenance.

The manifest distinguishes accounting feasibility from exact historical donor-field equivalence. No new accounting-formula implementation is included.
